# 01 — Text Preprocessing

Loads the three MELD CSV splits, normalises labels, builds the label map from
the **training split only** (no leakage), and encodes integer labels.

**Exports** (available to any notebook that does `%run 01_preprocess_text.ipynb`):
- `train_df`, `dev_df`, `test_df` — cleaned DataFrames
- `label2id`, `id2label`          — label ↔ int mappings
- `labels_sorted`                 — ordered list of emotion strings

In [ ]:
import os
import sys

# Make sure the project root (parent of notebooks/) is on the path so 'src' is importable.
_here = os.path.abspath(os.getcwd())
_root = _here if os.path.isdir(os.path.join(_here, 'src')) else os.path.abspath(os.path.join(_here, '..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

In [ ]:
from src.config import TRAIN_CSV, DEV_CSV, TEST_CSV, SEED
from src.data.text_preprocessing import load_split, build_label_map
from src.training.utils import set_seed

set_seed(SEED)

## Step 1 — Load train and build label map

We load the training split first (without encoding) so we can build the label
map exclusively from training labels. This prevents label leakage from dev/test.

In [ ]:
# Load train without encoding first — we need the raw emotion strings
train_df_raw = load_split(TRAIN_CSV)

# Build label map from train only
label2id, id2label = build_label_map(train_df_raw)
labels_sorted = list(label2id.keys())

print('Label map:', label2id)

## Step 2 — Load all splits with integer encoding

In [ ]:
train_df = load_split(TRAIN_CSV, label2id=label2id)
dev_df   = load_split(DEV_CSV,   label2id=label2id)
test_df  = load_split(TEST_CSV,  label2id=label2id)

print(f'Train: {len(train_df):,} | Dev: {len(dev_df):,} | Test: {len(test_df):,}')
print(f'Dropped from dev (unseen labels): {len(load_split(DEV_CSV)) - len(dev_df)}')
print(f'Dropped from test (unseen labels): {len(load_split(TEST_CSV)) - len(test_df)}')

## Step 3 — Sanity checks

In [ ]:
# Verify dtypes and that no NaNs slipped through
for name, df in [('train', train_df), ('dev', dev_df), ('test', test_df)]:
    assert df['label'].dtype == int, f'{name}: label dtype is {df["label"].dtype}'
    assert df.isna().sum().sum() == 0, f'{name}: contains NaN values'
    print(f'[{name}] OK — {len(df):,} rows, dtypes: {df.dtypes.to_dict()}')

In [ ]:
# Preview
train_df.head(8)